In [ ]:
import re
import time
import shutil
import unicodedata
import numpy as np
import pandas as pd
import json

from os import path
from pathlib import Path
from datetime import datetime
from sqlalchemy import create_engine, text, types as sat

🧩 Bloque 1 – Importación de librerías

En este bloque se importan las librerías necesarias para:
- Manipular archivos Excel y datos con pandas y numpy.
- Conectarse a la base de datos SQL Server mediante SQLAlchemy.
- Trabajar con fechas, rutas, archivos y expresiones regulares.

Estas herramientas permiten la automatización completa del proceso de lectura, limpieza y carga de datos en SQL Server.

In [ ]:
#from pathlib import Path
import sys
import importlib

# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks
# queremos:  .../traspaso-innominados/functions
PROJECT_ROOT = Path.cwd().resolve().parents[1]
FUNCTIONS_DIR = (PROJECT_ROOT / "functions").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
# Si tu archivo se llama functions.py dentro de FUNCTIONS_DIR, esto funciona.
# Si es paquete "functions/" con __init__.py, esto también funciona.
module_name = "functions"

if module_name in sys.modules:
    fun = importlib.reload(sys.modules[module_name])
else:
    fun = importlib.import_module(module_name)

print("Importado desde:", getattr(fun, "__file__", "<sin __file__>"))

🧩 Bloque 1.1 – Importación de configuraciones

In [ ]:
try:
    with open(path.join(PROJECT_ROOT, "config", "config_conosur.json"), 'r') as file:
        data = json.load(file)
    
except json.JSONDecodeError:
    print("Error: Failed to decode JSON from the file.")

🗄️ Bloque 2 – Conexión a SQL Server

En este bloque se configura la conexión al servidor SQL Server utilizando SQLAlchemy y el driver ODBC 17.

Se definen variables de conexión (server, database, schema, table).

Se crea el objeto engine, que será utilizado para ejecutar consultas e insertar datos.

Los parámetros fast_executemany=True y pool_pre_ping=True mejoran el rendimiento y evitan errores de conexión inactiva.

In [ ]:
# --- Parámetros ---
server = data["server_config"]["server"]
database = data["server_config"]["database"]
schema = data["server_config"]["schema"]

# --- Construir engine + obtener connection_string pyodbc ---
# Usa tus funciones: build_sqlserver_engine (y opcionalmente diagnósticos)

driver = "ODBC Driver 17 for SQL Server"  # o "ODBC Driver 18 for SQL Server"

# 1) ODBC connection string (pyodbc) -> para query_to_df / df_to_db / exec_sql
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# Si tu entorno requiere TLS/cert (muy común con Driver 18), descomenta:
# connection_string += "Encrypt=yes;TrustServerCertificate=yes;"

# 2) SQLAlchemy engine -> usando tu helper (hace test SELECT 1 y crea el engine)
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    # encrypt=True, trust_server_certificate=True,  # si lo necesitas
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise",  # "warn" si prefieres que no explote y solo imprima diagnóstico
)

📁 Bloque 3 – Configuración de ruta y hoja preferida

En este bloque se define la carpeta donde se buscarán los archivos Excel y se establecen las configuraciones básicas:
- RUTA_ARCHIVOS: ubicación de los archivos.
- EXTS: extensiones válidas (por defecto .xlsx).
- PREFERRED_SHEETS: hojas de cálculo preferidas que el script buscará automáticamente.

Esto permite encontrar y seleccionar automáticamente el archivo más reciente y la hoja de trabajo correcta.

In [ ]:
RUTA_ARCHIVOS = (PROJECT_ROOT / "data" / "CCLA" / "MARSH").resolve()


print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_ARCHIVOS:", RUTA_ARCHIVOS, "exists:", RUTA_ARCHIVOS.exists())

#RUTA_ARCHIVOS = Path(r"C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\CCLA\MARSH")
EXTS = (".xlsx",)  # Extensiones de archivo permitidas
#Se recomienda transformar el archivo ya que habitualmente llega de forma xlsb, transformarlo a xlsx

PREFERRED_SHEETS = ["base", "Base cesantia SURA Stock", "Base"]
FALLBACK_SHEET_INDEX = 0

📄 Bloque 5 – Selección del archivo más reciente y hoja de trabajo

En este bloque se identifica el archivo Excel más reciente dentro de la carpeta configurada.
- Se ordenan los archivos por fecha de modificación.
- Se selecciona la hoja de trabajo más adecuada según los nombres definidos.
- Si el archivo está bloqueado por OneDrive, se crea una copia temporal para poder leerlo.

Esto asegura que el proceso siempre trabaje con el archivo más actualizado.

In [5]:
assert RUTA_ARCHIVOS.is_dir(), f"No existe carpeta: {RUTA_ARCHIVOS}"
archivos = [p for p in RUTA_ARCHIVOS.iterdir() if p.is_file() and p.suffix.lower() in EXTS]
archivos.sort(key=lambda p: p.stat().st_mtime, reverse=True)
assert archivos, "No se encontraron Excel válidos."

archivo_origen = archivos[0]
print(f"📄 Archivo más reciente: {archivo_origen.name}  |  {datetime.fromtimestamp(archivo_origen.stat().st_mtime):%Y-%m-%d %H:%M:%S}")

_tmp_copy_path = None

def _norm(s): 
    return " ".join(str(s).split()).lower()

pref_norm = {_norm(n) for n in PREFERRED_SHEETS}

try:
    try:
        with pd.ExcelFile(archivo_origen, engine="openpyxl") as xls:
            print("Hojas:", xls.sheet_names)

            target = None
            for s in xls.sheet_names:
                if _norm(s) in pref_norm:
                    target = s
                    break
            if target is None:
                target = xls.sheet_names[FALLBACK_SHEET_INDEX]
            print(f"✅ Hoja objetivo: {target}")

    except PermissionError:
        _tmp_copy_path = archivo_origen.parent / f"__tmp_copy_{int(time.time()*1000)}{archivo_origen.suffix}"
        shutil.copy2(archivo_origen, _tmp_copy_path)
        print(f"📂 No se pudo abrir el archivo original, usando copia temporal: {_tmp_copy_path.name}")

        with pd.ExcelFile(_tmp_copy_path, engine="openpyxl") as xls:
            print("Hojas:", xls.sheet_names)

            target = None
            for s in xls.sheet_names:
                if _norm(s) in pref_norm:
                    target = s
                    break
            if target is None:
                target = xls.sheet_names[FALLBACK_SHEET_INDEX]
            print(f"✅ Hoja objetivo (copia): {target}")

except Exception as e:
    raise SystemExit(f"❌ Error al abrir el Excel: {e}")

📄 Archivo más reciente: SC202511_Ok Sura.xlsx  |  2026-01-19 16:45:02
Hojas: ['base', 'Resumen']
✅ Hoja objetivo: base


📊 Bloque 6 – Lectura segura del Excel y limpieza de encabezados

En este bloque se realiza la lectura del archivo Excel seleccionado y la limpieza inicial de los datos:
- Se utiliza pd.read_excel(..., dtype=str) para mantener el control total de los tipos de datos.
- Se eliminan valores vacíos y se normalizan los encabezados con la función fun.normalize_name.
- Se imprime una vista previa del DataFrame resultante.

El resultado es un DataFrame uniforme y listo para su mapeo.

In [ ]:
def read_excel_safe(io, sheet_name):
    try:
        return pd.read_excel(io, sheet_name=sheet_name, dtype=str, engine="openpyxl")
    except Exception as e:
        raise SystemExit(f"No se pudo leer la hoja '{sheet_name}': {e}")

io = _tmp_copy_path if _tmp_copy_path else archivo_origen
df_raw = read_excel_safe(io, target)

for c in df_raw.columns:
    if df_raw[c].dtype == object:
        df_raw[c] = df_raw[c].astype(str).str.strip().replace({"None": "", "nan": "", "NaN": ""})
df_raw.columns = [fun.normalize_name(c) for c in df_raw.columns]
print("Encabezados normalizados:", list(df_raw.columns))
df_raw.head()

Encabezados normalizados: ['foliocredito', 'rutafiliado', 'dvafiliado', 'nombreafiliado', 'plazo', 'montobruto', 'fecotorgamiento', 'fechaprimervto', 'fechaultimovto', 'valorcuota', 'fechaprima', 'prima', 'desgravamen', 'fechadefuncion', 'origendefuncion', 'producto', 'folioorigen', 'fechaorigen', 'tasaorigen', 'corredora', 'unnamed_20', 'fecha_de_venta', 'fecha_de_baja', 'poliza', 'tasa', 'prima_bruta_mensual', 'iva', 'prima_neta', 'diferencia_ccla']


,foliocredito,rutafiliado,dvafiliado,nombreafiliado,plazo,montobruto,fecotorgamiento,fechaprimervto,fechaultimovto,valorcuota,...,corredora,unnamed_20,fecha_de_venta,fecha_de_baja,poliza,tasa,prima_bruta_mensual,iva,prima_neta,diferencia_ccla
0,173000044182,19778569,1,DAVID MUNOZ MOLINA,18,304172,20250408,20250531,20261031,23830,...,Marsh,,2025-04-08 00:00:00,,7561166,0.002,608,97.07563025210084,510.92436974789916,0
1,41000406832,20092283,2,CAMILA ARAYA ARAYA,36,2622983,20231002,20231130,20261031,125469,...,Marsh,,2023-10-02 00:00:00,,7561166,0.002,5246,837.5966386554619,4408.403361344538,0
2,84000896599,17870326,9,FRANCISCO RIVAS GALLARDO,30,2925403,20250326,20250430,20270930,156787,...,Marsh,,2025-03-26 00:00:00,,7561166,0.002,5851,934.1932773109238,4916.806722689076,0
3,23000473282,12423631,2,MARITZA SAIRES MENDOZA,12,1010584,20250926,20251031,20260930,105462,...,Marsh,,2025-09-26 00:00:00,,7561166,0.002,2021,322.68067226890753,1698.3193277310925,0
4,137000199854,11739366,6,PAULA MORALES ROMERO,10,485261,20250717,20250831,20260531,59334,...,Marsh,,2025-07-17 00:00:00,,7561166,0.002,971,155.03361344537814,815.9663865546219,0


🧱 Bloque 7 – Mapeo de columnas origen → destino

En este bloque se mapean las columnas originales del Excel hacia las columnas estándar del sistema:
- Se utiliza la función pick() para tolerar cambios o variaciones en los encabezados.
- Se construye el DataFrame df_can con todas las columnas requeridas.
- Se agrega la columna Nombre_de_archivo para identificar la fuente del dato.

Este paso genera una estructura homogénea y compatible con la base de datos.

In [7]:
def pick(df, *names):
    for n in names:
        if n in df.columns:
            return df[n]
    return pd.Series([None]*len(df), index=df.index)


# Origen → Destino
foliocredito        = pick(df_raw, "n_operacion", "no_operacion", "noperacion", "operacion", "folio", "foliocredito")
rutafiliado         = pick(df_raw, "afirut", "rut_afiliado", "rut", "rutafiliado")
dvafiliado          = pick(df_raw, "afirutdv", "dv_afiliado", "dv", "dvafiliado")
NombreAfiliado      = pick(df_raw, "afinom", "nombre_afiliado", "nombre", "nombreafiliado")
Plazo               = pick(df_raw, "crecuotot", "plazo")
MontoBruto          = pick(df_raw, "cresolmon", "monto_bruto", "monto", "montobruto")
fecotorgamiento     = pick(df_raw, "fecotorgamiento", "fecha_otorgamiento", "fec_otorgamiento")
fechaPrimerVto      = pick(df_raw, "fecinicob", "fecha_primer_vto", "fec_ini_cob", "fechaprimervto")
FechaUltimoVto      = pick(df_raw, "fectercob", "fecha_ultimo_vto", "fec_ter_cob", "fechaultimovto")
ValorCuota          = pick(df_raw, "crecuomon", "valor_cuota", "valorcuota")
FechaPrima          = pick(df_raw, "fechaprima", "fecha_prima")
Prima               = pick(df_raw, "prima_cliente", "prima_bruta", "prima_cliente_" , "prima")
Desgravamen         = pick(df_raw, "prima_desgravamen", "desgravamen")
FechaDefuncion      = pick(df_raw, "fecdefuncion", "fecha_defuncion", "fechadefuncion")
OriginDefuncion     = pick(df_raw, "origen_defuncion", "origin_defuncion", "origendefuncion")
Producto            = pick(df_raw, "producto")
FolioOrigen         = pick(df_raw, "folio_original", "folio_origen", "folioorigen")
FechaOrigen         = pick(df_raw, "fecha_otorgamiento_folio_original", "fecha_origen", "fechaorigen")
TasaOrigen          = pick(df_raw, "tasa_interes", "tasaorigen")
Corredora           = pick(df_raw, "corredora")
Fecha_de_venta      = pick(df_raw, "fecha_de_venta", "fechadeventa", "fecha_venta")
Fecha_de_baja       = pick(df_raw, "fecha_de_baja", "fechadebaja", "fecha_baja")
Poliza              = pick(df_raw, "poliza", "póliza")
Tasa                = pick(df_raw, "tasa_interes", "tasa")
PrimaBruta          = pick(df_raw, "prima_bruta_mensual", "prima_bruta")
Iva                 = pick(df_raw, "iva")
PrimaNeta           = pick(df_raw, "primaneta", "prima_neta")
Diferencia_CCLA     = pick(df_raw, "diferencia_ccla", "diferenciaccla")


df_can = pd.DataFrame({
    "foliocredito": foliocredito,
    "rutafiliado": rutafiliado,
    "dvafiliado": dvafiliado,
    "NombreAfiliado": NombreAfiliado,
    "Plazo": Plazo,
    "MontoBruto": MontoBruto,
    "fecotorgamiento": fecotorgamiento,
    "fechaPrimerVto": fechaPrimerVto,
    "FechaUltimoVto": FechaUltimoVto,
    "ValorCuota": ValorCuota,
    "FechaPrima": FechaPrima,
    "Prima": Prima,
    "Desgravamen": Desgravamen,
    "FechaDefuncion": FechaDefuncion,
    "OrigenDefuncion": OriginDefuncion,
    "Producto": Producto,
    "FolioOrigen": FolioOrigen,
    "FechaOrigen": FechaOrigen,
    "TasaOrigen": TasaOrigen,
    "Corredora": Corredora,
    "FECHA_DE_VENTA": Fecha_de_venta,
    "FECHA_DE_BAJA": Fecha_de_baja,
    "Poliza": Poliza,
    "Tasa": Tasa,
    "Prima_Bruta_mensual": PrimaBruta,
    "IVA": Iva,
    "Prima_Neta": PrimaNeta,
    "Diferencia_CCLA": Diferencia_CCLA,
    "Nombre_de_archivo": archivo_origen.name,
})

df_can.head()

,foliocredito,rutafiliado,dvafiliado,NombreAfiliado,Plazo,MontoBruto,fecotorgamiento,fechaPrimerVto,FechaUltimoVto,ValorCuota,...,Corredora,FECHA_DE_VENTA,FECHA_DE_BAJA,Poliza,Tasa,Prima_Bruta_mensual,IVA,Prima_Neta,Diferencia_CCLA,Nombre_de_archivo
0,173000044182,19778569,1,DAVID MUNOZ MOLINA,18,304172,20250408,20250531,20261031,23830,...,Marsh,2025-04-08 00:00:00,,7561166,0.002,608,97.07563025210084,510.92436974789916,0,SC202511_Ok Sura.xlsx
1,41000406832,20092283,2,CAMILA ARAYA ARAYA,36,2622983,20231002,20231130,20261031,125469,...,Marsh,2023-10-02 00:00:00,,7561166,0.002,5246,837.5966386554619,4408.403361344538,0,SC202511_Ok Sura.xlsx
2,84000896599,17870326,9,FRANCISCO RIVAS GALLARDO,30,2925403,20250326,20250430,20270930,156787,...,Marsh,2025-03-26 00:00:00,,7561166,0.002,5851,934.1932773109238,4916.806722689076,0,SC202511_Ok Sura.xlsx
3,23000473282,12423631,2,MARITZA SAIRES MENDOZA,12,1010584,20250926,20251031,20260930,105462,...,Marsh,2025-09-26 00:00:00,,7561166,0.002,2021,322.68067226890753,1698.3193277310925,0,SC202511_Ok Sura.xlsx
4,137000199854,11739366,6,PAULA MORALES ROMERO,10,485261,20250717,20250831,20260531,59334,...,Marsh,2025-07-17 00:00:00,,7561166,0.002,971,155.03361344537814,815.9663865546219,0,SC202511_Ok Sura.xlsx


🕓 Bloque 8 – Extracción de período y nombre canónico

En este bloque se extrae el período (YYYYMM) desde el nombre del archivo y se genera un nombre canónico estandarizado:
- La función _extract_yyyymm_from_name() detecta el mes y año.
- canonicalizar_planes_safe() asegura que el proceso no se detenga si no se encuentra el período.
- Se reemplaza Nombre_de_archivo por el nombre canónico (SC_MARSHYYYYMM.xlsx).

Esto permite identificar los archivos de manera única y ordenada dentro de la base de datos.

In [ ]:
MESES = {
    "ENERO": "01", "ENE": "01",
    "FEBRERO": "02", "FEB": "02",
    "MARZO": "03", "MAR": "03",
    "ABRIL": "04", "ABR": "04",
    "MAYO": "05", "MAY": "05",
    "JUNIO": "06", "JUN": "06",
    "JULIO": "07", "JUL": "07",
    "AGOSTO": "08", "AGO": "08",
    "SEPTIEMBRE": "09", "SETIEMBRE": "09", "SEP": "09", "SET": "09",
    "OCTUBRE": "10", "OCT": "10",
    "NOVIEMBRE": "11", "NOV": "11",
    "DICIEMBRE": "12", "DIC": "12",
}


def _extract_yyyymm_from_name(nombre: str) -> str:
    """
    Intenta extraer YYYYMM desde el nombre del archivo (sin importar la extensión).
    Cubre:
      - YYYYMM
      - YYYY[-_/ .]?MM
      - MM[-_/ .]?YYYY
      - MES(ES) + YYYY (con tildes/abreviaturas) en cualquier orden
    Lanza ValueError si no encuentra período.
    """
    stem = Path(nombre).stem
    stem_norm = fun._strip_accents(stem).upper()

    m = re.search(r"(20\d{2})(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

    m = re.search(r"(20\d{2})[-_/.\s]?(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

    m = re.search(r"(0[1-9]|1[0-2])[-_/.\s]?(20\d{2})", stem_norm)
    if m:
        return f"{m.group(2)}{m.group(1)}"

    m_year = re.search(r"(20\d{2})", stem_norm)
    if m_year:
        year = m_year.group(1)
        # Aseguramos coincidencias completas de palabra para evitar falsos positivos (e.g. "MAR" en "MARCHA")
        for mes_txt, mm in MESES.items():
            # \b no siempre funciona con underscores, por eso agregamos delimitadores comunes
            if re.search(rf"(?<![A-Z0-9]){mes_txt}(?![A-Z0-9])", stem_norm):
                return f"{year}{mm}"

    raise ValueError(f"No pude extraer YYYYMM desde el nombre: {nombre}")


def canonicalizar_planes(nombre: str) -> str:
    """
    Devuelve un nombre canónico estandarizado: 'SC_MARSHYYYYMM.xlsx'
    Si no logra extraer el período (YYYYMM), devuelve un nombre genérico con aviso.
    """
    try:
        yyyymm = _extract_yyyymm_from_name(nombre)
        return f"SC_MARSH{yyyymm}.xlsx"
    except ValueError as e:
        print(f"⚠️ No se pudo extraer período desde el nombre '{nombre}'. "
              f"Se usará un nombre genérico. Detalle: {e}")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        return f"SC_MARSH{timestamp}.xlsx"


nombre_canonico = canonicalizar_planes(archivo_origen.name)
print("Nombre original :", archivo_origen.name)
print("Nombre canónico :", nombre_canonico)


df_can["Nombre_de_archivo"] = nombre_canonico

Nombre original : SC202511_Ok Sura.xlsx
Nombre canónico : SC_MARSH202511.xlsx


🧹 Bloque 9 – Limpieza de filas de cierre (TOTAL o separadores)

En este bloque se eliminan las filas innecesarias al final del Excel, como totales o separadores vacíos:
- Se detectan filas con alto porcentaje de valores nulos.
- Se comparan sumas para confirmar si corresponde a una fila de “TOTAL”.
- Se eliminan automáticamente para evitar duplicar datos.

El resultado es un DataFrame limpio y sin filas redundantes.

In [ ]:
def _is_nullish(v):
    if pd.isna(v):
        return True
    if isinstance(v, str):
        s = v.strip()
        return (s == "") or (s.lower() == "none")
    return False


def _row_nullish_ratio(row: pd.Series, exclude=()):
    cols = [c for c in row.index if c not in exclude]
    if not cols:
        return 1.0
    n_null = sum(_is_nullish(row[c]) for c in cols)
    return n_null / len(cols)


def drop_trailing_mostly_null(
    df: pd.DataFrame,
    null_check_exclude=("Nombre_de_archivo",),
    also_exclude_money_cols=("Prima_Bruta_mensual","IVA","Prima_Neta","Diferencia_CCLA"),
    null_ratio_threshold=0.80,
    verbose=True,
):
    """
    Elimina filas al final del DF mientras tengan un ratio alto de nulos.
    Además MUESTRA las filas eliminadas con todo su contenido.
    """
    if df is None or df.empty:
        if verbose:
            print("DF vacío: nada que hacer.")
        return df

    out = df.copy()
    removed_rows = [] 

    exclude = set(null_check_exclude) | set(also_exclude_money_cols)

    i = len(out) - 1
    while i >= 0:
        row = out.iloc[i]
        ratio = _row_nullish_ratio(row, exclude=exclude)

        if verbose:
            print(f"Fila índice {out.index[i]} → null_ratio={ratio:.2%}")

        if ratio >= null_ratio_threshold:
            # Guardar la fila completa ANTES de eliminarla
            removed_rows.append((out.index[i], row.copy()))

            i -= 1
        else:
            break

    # Si no hubo nada para eliminar
    if not removed_rows:
        if verbose:
            print("❎ No se detectaron filas finales mayoritariamente nulas.")
        return out

    # Mostrar filas eliminadas
    if verbose:
        print(f"\n🧹 Eliminando {len(removed_rows)} fila(s) finales mayoritariamente nulas:\n")
        for idx, row in removed_rows:
            print(f"--- Fila eliminada (índice original {idx}) ---")
            print(row.to_frame().T)   # muestra la fila completa en formato bonito
            print("\n")

    # Finalmente eliminar del DF
    drop_indices = [idx for idx, _ in removed_rows]
    out = out.drop(index=drop_indices).reset_index(drop=True)

    return out


df_can = drop_trailing_mostly_null(
    df_can,
    null_check_exclude=("Nombre_de_archivo",),
    also_exclude_money_cols=("Prima_Bruta_mensual","IVA","Prima_Neta","Diferencia_CCLA"),
    null_ratio_threshold=0.80,
    verbose=True
)


Fila índice 101644 → null_ratio=95.83%
Fila índice 101643 → null_ratio=91.67%
Fila índice 101642 → null_ratio=8.33%

🧹 Eliminando 2 fila(s) finales mayoritariamente nulas:

--- Fila eliminada (índice original 101644) ---
       foliocredito rutafiliado dvafiliado NombreAfiliado Plazo MontoBruto  \
101644                                                                       

       fecotorgamiento fechaPrimerVto FechaUltimoVto ValorCuota  ...  \
101644                                                           ...   

       Corredora FECHA_DE_VENTA FECHA_DE_BAJA Poliza Tasa Prima_Bruta_mensual  \
101644                  TOTALES                                     492627458   

                     IVA         Prima_Neta Diferencia_CCLA  \
101644  78654804.2184829  413972653.7815718            5087   

          Nombre_de_archivo  
101644  SC_MARSH202511.xlsx  

[1 rows x 29 columns]


--- Fila eliminada (índice original 101643) ---
       foliocredito rutafiliado dvafiliado NombreAfili

🧮 Bloque 10 – Tipado de columnas y normalización de fechas

En este bloque se asignan los tipos de datos correctos a cada columna y se normalizan las fechas al formato YYYYMMDD:
- Se convierten valores numéricos, decimales y fechas de Excel.
- Se ajustan los tipos (Int64, float64, string).
- Se truncan las cadenas a la longitud máxima esperada por la base de datos.

Este paso asegura que los datos sean compatibles con el esquema de SQL Server

In [10]:
INT_COLS    = ["rutafiliado", "Plazo", "MontoBruto", "fecotorgamiento","fechaPrimerVto", "FechaUltimoVto", "ValorCuota","FechaPrima",
                "Prima", "Desgravamen", "FechaDefuncion", "FechaOrigen", "Poliza"]
BIGINT_COLS = ["foliocredito", "FolioOrigen"]
FLOAT_COLS  = ["TasaOrigen", "Tasa", "Prima_Bruta_mensual", "IVA", "Prima_Neta", "Diferencia_CCLA"]
STR_COLS    = ["NombreAfiliado", "OrigenDefuncion", "Producto", "Corredora", "Nombre_de_archivo"]
DV_COL      = "dvafiliado"
DATE_COLS   = ["FECHA_DE_VENTA","FECHA_DE_BAJA"]


def _to_num_series(s: pd.Series) -> pd.Series:
    if not pd.api.types.is_object_dtype(s):
        return pd.to_numeric(s, errors="coerce")
    s2 = s.astype(str).str.strip().replace({"": np.nan, "None": np.nan, "none": np.nan})
    return pd.to_numeric(s2, errors="coerce")


def _excel_serial_to_datetime(x):
    """
    Convierte números 'serial Excel' a datetime (base 1899-12-30), deja el resto intacto.
    """
    if pd.isna(x):
        return np.nan
    # Si es numérico y razonable como serial de Excel
    try:
        # cuidado con strings numéricas
        xv = float(x)
        if 1 <= xv <= 60000:  # ~año 2064
            return pd.Timestamp("1899-12-30") + pd.to_timedelta(int(xv), unit="D")
    except Exception:
        pass
    return x


def _parse_to_yyyymmdd_int(series: pd.Series) -> pd.Series:
    """
    1) Limpia espacios
    2) Convierte serial Excel -> datetime si aplica
    3) pd.to_datetime con errors='coerce'
    4) Formatea YYYYMMDD y castea a Int64 (preserva <NA> cuando no se puede parsear)
    """
    s = series.copy()
    # Si viene como object, sanea
    if pd.api.types.is_object_dtype(s):
        s = s.astype(str).str.strip().replace({"": np.nan, "None": np.nan, "none": np.nan})
    # Intentar serial Excel
    s = s.map(_excel_serial_to_datetime)
    # Parse genérico (maneja "2025-01-16 00:00:00", "16/01/2025", etc.)
    d = pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)
    # A YYYYMMDD como string, luego Int64
    out = d.dt.strftime("%Y%m%d")
    out = pd.to_numeric(out, errors="coerce").astype("Int64")
    return out


# --- 0) Primero: convertir fechas a YYYYMMDD (Int64) ---
for c in DATE_COLS:
    if c in df_can.columns:
        df_can[c] = _parse_to_yyyymmdd_int(df_can[c])

# BIGINT -> pandas Int64 (acepta NaN)
for c in BIGINT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("Int64")

# INT -> pandas Int64
for c in INT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("Int64")

# FLOAT -> float64
for c in FLOAT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("float64")

# DV: char(1) en mayúscula
if DV_COL in df_can.columns:
    df_can[DV_COL] = (
        df_can[DV_COL]
        .astype("string")
        .str.strip()
        .str.upper()
        .map(lambda x: x[:1] if pd.notna(x) and len(x) > 0 else pd.NA)
    )

if "NombreAfiliado" in df_can.columns:
    df_can["NombreAfiliado"] = df_can["NombreAfiliado"].astype("string").str.strip().str.slice(0, 50)

if "OrigenDefuncion" in df_can.columns:
    df_can["OrigenDefuncion"] = df_can["OrigenDefuncion"].astype("string").str.strip().str.slice(0, 50)

if "Producto" in df_can.columns:
    df_can["Producto"] = df_can["Producto"].astype("string").str.strip().str.slice(0, 50)

if "Corredora" in df_can.columns:
    df_can["Corredora"] = df_can["Corredora"].astype("string").str.strip().str.slice(0, 50)

if "Nombre_de_archivo" in df_can.columns:
    df_can["Nombre_de_archivo"] = df_can["Nombre_de_archivo"].astype("string").str.strip().str.slice(0, 50)

print("✅ dtypes DESPUÉS:\n", df_can.dtypes)

# --- 3) (Opcional) Validación rápida de nulos en columnas críticas ---
criticas = ["foliocredito","rutafiliado","dvafiliado","Nombre_de_archivo"]
presentes = [c for c in criticas if c in df_can.columns]
if presentes:
    print("\n🔎 Nulos en columnas críticas:")
    for c in presentes:
        print(f" - {c}: {int(df_can[c].isna().sum())} nulos")

cols_sql = [
    "foliocredito","rutafiliado","dvafiliado","NombreAfiliado","Plazo","MontoBruto","fecotorgamiento",
    "fechaPrimerVto","FechaUltimoVto","ValorCuota","FechaPrima","Prima","Desgravamen","FechaDefuncion",
    "OrigenDefuncion","Producto","FolioOrigen", "FechaOrigen","TasaOrigen", "Corredora", "FECHA_DE_VENTA","FECHA_DE_BAJA",
    "Poliza","Tasa","Prima_Bruta_mensual","IVA","Prima_Neta","Diferencia_CCLA","Nombre_de_archivo"
]

df_sql = df_can[[c for c in cols_sql if c in df_can.columns]].copy()

print("\n📦 df_sql listo con columnas:", list(df_sql.columns))

C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_15784\2936138103.py:48: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  d = pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_15784\2936138103.py:48: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  d = pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)


✅ dtypes DESPUÉS:
 foliocredito                    Int64
rutafiliado                     Int64
dvafiliado                     object
NombreAfiliado         string[python]
Plazo                           Int64
MontoBruto                      Int64
fecotorgamiento                 Int64
fechaPrimerVto                  Int64
FechaUltimoVto                  Int64
ValorCuota                      Int64
FechaPrima                      Int64
Prima                           Int64
Desgravamen                     Int64
FechaDefuncion                  Int64
OrigenDefuncion        string[python]
Producto               string[python]
FolioOrigen                     Int64
FechaOrigen                     Int64
TasaOrigen                    float64
Corredora              string[python]
FECHA_DE_VENTA                  Int64
FECHA_DE_BAJA                   Int64
Poliza                          Int64
Tasa                          float64
Prima_Bruta_mensual           float64
IVA                           f

🔎 Bloque 11 – Validación de NOMBRE_ARCHIVO y conteo de filas por archivo

Este bloque valida:
- Que NOMBRE_ARCHIVO exista.
- Que tenga valores válidos (no vacíos).
- Cuenta cuántas filas trae cada archivo en el DataFrame.

Genera un diccionario {archivo → cantidad de filas}, usado luego para control de versiones.

In [11]:
assert "Nombre_de_archivo" in df_sql.columns, "Falta la columna 'Nombre_de_archivo' en df_sql."

df_sql["Nombre_de_archivo"] = (
    df_sql["Nombre_de_archivo"]
    .astype("string")
    .str.strip()
)

vals = [
    v for v in df_sql["Nombre_de_archivo"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚨 No se encontraron valores válidos en 'Nombre_de_archivo'.")

counts = (
    df_sql["Nombre_de_archivo"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"📄 Se detectaron {len(vals)} Nombre_de_archivo distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

📄 Se detectaron 1 Nombre_de_archivo distintos en el df_sql:
   - SC_MARSH202511.xlsx: 101643 filas


🚀 Bloque 12 – Carga con reemplazo selectivo por archivo (ETL transaccional)

Este bloque implementa el core del proceso ETL:
- Se ejecuta una sola transacción.
- Para cada NOMBRE_ARCHIVO:
- Consulta si existe data previa en SQL.
- Si existe → elimina solo ese archivo (reemplazo controlado).
- Inserta solo las filas nuevas.
- Imprime un resumen:
    - Filas reemplazadas
    - Filas insertadas
    - Archivos nuevos vs antiguos

Es un flujo robusto tipo "upsert por archivo".

In [ ]:
resumen = []

with engine.begin() as conn:
    qry_count = text(f"""
        SELECT 
            COUNT(*) AS cnt
        FROM {data["tablas_remotas"]["marsh_acumulado_r_bkp"]}
        WHERE 
            Nombre_de_archivo = :nombre
    """)

    qry_del = text(f"""
        DELETE FROM {data["tablas_remotas"]["marsh_acumulado_r_bkp"]}
        WHERE 
            Nombre_de_archivo = :nombre
    """)

    for nombre_archivo in vals:
        df_sub = df_sql[df_sql["Nombre_de_archivo"] == nombre_archivo]

        if df_sub.empty:
            print(f"⚠️ No se encontraron filas en df_sql para Nombre_de_archivo = '{nombre_archivo}'. Se omite.")
            continue

        existing_count = conn.execute(qry_count, {"nombre": nombre_archivo}).scalar() or 0

        if existing_count > 0:
            print(f"♻️ Se encontró data previa para Nombre_de_archivo = '{nombre_archivo}' "
                  f"({existing_count} filas en {data["tablas_remotas"]["marsh_acumulado_r_bkp"]}).")
            print("🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...")

            deleted = conn.execute(qry_del, {"nombre": nombre_archivo}).rowcount
            print(f"✅ Filas eliminadas en destino para '{nombre_archivo}': {deleted}")
            accion = "reemplazo"
        else:
            print(f"🆕 No hay data previa para Nombre_de_archivo = '{nombre_archivo}'. "
                  "Se insertará como archivo nuevo.")
            accion = "inserción nueva"

        df_sub.to_sql(
            name=data["tablas_remotas"]["marsh_acumulado_r_bkp"],
            con=conn,
            schema=schema,
            if_exists='append',
            index=False
        )

        filas_sub = len(df_sub)
        print(f"📥 Insertadas {filas_sub} filas para Nombre_de_archivo = '{nombre_archivo}'.")
        resumen.append((nombre_archivo, filas_sub, existing_count, accion))


print("\n📊 Resumen de carga por Nombre_de_archivo:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (archivo nuevo).")


♻️ Se encontró data previa para Nombre_de_archivo = 'SC_MARSH202511.xlsx' (101643 filas en dbo.MARSH_ACUMULADO_R).
🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...
✅ Filas eliminadas en destino para 'SC_MARSH202511.xlsx': 101643


🗑️ Bloque 13 – Eliminación del archivo procesado

Finalmente:
- Una vez cargado en SQL, el archivo fuente se elimina automáticamente.
- Maneja errores comunes:
    - Archivo en uso por Excel/OneDrive
    - Permisos insuficientes
    - Casos en que el archivo no existe
    
Cierra el ciclo ETL manteniendo la carpeta siempre limpia.

In [84]:
try:
    if archivo_origen.exists() and archivo_origen.is_file():
        archivo_origen.unlink()
        print(f"\n🗑️ Archivo eliminado correctamente: {archivo_origen}")
    else:
        print(f"\n⚠️ No se pudo borrar el archivo porque no es un archivo válido: {archivo_origen}")
except PermissionError:
    print(f"\n⚠️ No se pudo borrar '{archivo_origen}': está en uso por OneDrive o Excel.")
except Exception as e:
    print(f"\n⚠️ Error inesperado al borrar archivo '{archivo_origen}': {e}")


🗑️ Archivo eliminado correctamente: C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\CCLA\MARSH\SC202508_Ok Sura.xlsx


# Sección SQL

## VALIDACIONES / INSPECCIÓN

In [ ]:

query_1 = f"""
-- [MARSH] Muestra rápida
SELECT TOP 10 
    * 
FROM {data["tablas_remotas"]["marsh_acumulado_r_bkp"]};
"""
fun.exec_sql(query_1, connection_string)


In [ ]:
query_2 = f"""
-- [MARSH] Archivos cargados (desc)
SELECT DISTINCT 
    Nombre_de_archivo
FROM {data["tablas_remotas"]["marsh_acumulado_r_bkp"]}
ORDER BY 
    Nombre_de_archivo DESC;
"""

fun.exec_sql(query_2, connection_string)

## CONSTRUCCIÓN DE BASES FINALES POR CORREDOR / FUENTE

In [ ]:
query_3 = f"""
DROP TABLE IF EXISTS {data["tablas_remotas"]["marsh_final_acumulado_bkp"]};
"""

fun.exec_sql(query_3, connection_string)

In [ ]:
query_4 = f"""
SELECT *,
       /* 7561166 AS POLIZA, -- comentado en tu script */
       SUBSTRING(Nombre_de_archivo,
                 LEN(Nombre_de_archivo) - CHARINDEX('.', REVERSE(Nombre_de_archivo)) - 5,
                 6) AS MES_Recaudacion,
       8285 AS PLAN_TECNICO,
       4 AS PLAZO_CUOTAS,
       'Credito Consumo' AS Negocio
INTO {data["tablas_remotas"]["marsh_final_acumulado_bkp"]}
FROM {data["tablas_remotas"]["marsh_acumulado_r_bkp"]};
"""

fun.exec_sql(query_4, connection_string)

In [ ]:
query_5 = f"""
-- Ajuste provisorio: partner atrasado ~2 meses entre recaudación y real
UPDATE {data["tablas_remotas"]["marsh_final_acumulado_bkp"]}
SET 
    MES_Recaudacion = CASE
                        WHEN RIGHT(CAST(MES_Recaudacion AS VARCHAR(6)), 2) = '01' THEN (MES_Recaudacion - 89) -- enero -> año-1 mes12
                        ELSE (MES_Recaudacion - 1)
                      END;
"""

fun.exec_sql(query_5, connection_string)